# ResilientRoute – Vertex AI Risk Prediction Model
## Disruption Delay Probability Model for Indian Supply Chains

This notebook trains a lightweight XGBoost model to predict shipment delay probability
based on origin, destination, lane weather stress, and port congestion proxies.
The model is then deployed to a Vertex AI Endpoint for real-time inference.

In [ ]:
%pip install google-cloud-bigquery google-cloud-aiplatform xgboost scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb
import json, os

PROJECT_ID = os.getenv('GCP_PROJECT_ID', 'resilientroute-demo')
REGION = os.getenv('GCP_REGION', 'asia-south1')
BQ_DATASET = 'resilientroute'

## 1. Generate Synthetic Training Data
We create realistic training data based on Indian trade lanes.
In production, this would come from BigQuery historical events.

In [ ]:
np.random.seed(42)
N = 5000

lanes = {
    'lane-1': {'origin': 'Mumbai', 'dest': 'Singapore', 'base_risk': 0.45},
    'lane-2': {'origin': 'Mundra', 'dest': 'Dubai', 'base_risk': 0.25},
    'lane-3': {'origin': 'Chennai', 'dest': 'Singapore', 'base_risk': 0.15},
    'lane-4': {'origin': 'Kochi', 'dest': 'Salalah', 'base_risk': 0.30},
    'lane-5': {'origin': 'Visakhapatnam', 'dest': 'Kolkata', 'base_risk': 0.20},
    'lane-6': {'origin': 'Delhi', 'dest': 'Dubai', 'base_risk': 0.35},
    'lane-7': {'origin': 'Chennai', 'dest': 'Colombo', 'base_risk': 0.50},
    'lane-8': {'origin': 'Paradip', 'dest': 'Chittagong', 'base_risk': 0.30},
    'lane-9': {'origin': 'Bengaluru', 'dest': 'Tuticorin', 'base_risk': 0.10},
    'lane-10': {'origin': 'Mumbai', 'dest': 'Dubai', 'base_risk': 0.30},
}

data = []
for _ in range(N):
    lane_id = np.random.choice(list(lanes.keys()))
    lane = lanes[lane_id]
    
    weather_stress = np.clip(lane['base_risk'] + np.random.normal(0, 0.15), 0, 1)
    is_cyclone_season = np.random.choice([0, 1], p=[0.6, 0.4])
    port_congestion = np.random.uniform(0.1, 0.9)
    cargo_value_cr = np.random.uniform(0.5, 50)
    speed_knots = np.random.uniform(8, 22)
    progress_pct = np.random.uniform(5, 95)
    
    # Delay probability model
    delay_prob = (
        0.3 * weather_stress +
        0.2 * is_cyclone_season * weather_stress +
        0.2 * port_congestion +
        0.1 * (1 - progress_pct/100) +
        0.05 * max(0, 15 - speed_knots) / 15 +
        np.random.normal(0, 0.05)
    )
    delay_prob = np.clip(delay_prob, 0, 1)
    is_delayed = 1 if delay_prob > 0.5 else 0
    
    data.append({
        'lane_id': lane_id,
        'weather_stress': round(weather_stress, 3),
        'is_cyclone_season': is_cyclone_season,
        'port_congestion': round(port_congestion, 3),
        'cargo_value_cr': round(cargo_value_cr, 2),
        'speed_knots': round(speed_knots, 1),
        'progress_pct': round(progress_pct, 1),
        'delay_prob': round(delay_prob, 3),
        'is_delayed': is_delayed,
    })

df = pd.DataFrame(data)
print(f'Dataset: {len(df)} rows')
print(f'Delayed: {df.is_delayed.mean():.1%}')
df.head()

## 2. Feature Engineering & Training

In [ ]:
# One-hot encode lane_id
df_encoded = pd.get_dummies(df, columns=['lane_id'], drop_first=True)

feature_cols = [c for c in df_encoded.columns if c not in ['is_delayed', 'delay_prob']]
X = df_encoded[feature_cols]
y = df_encoded['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# Evaluate
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f'\nAUC-ROC: {auc:.4f}')
print(classification_report(y_test, model.predict(X_test)))

## 3. Save Model Artifact

In [ ]:
import joblib
os.makedirs('model_artifacts', exist_ok=True)
model.save_model('model_artifacts/risk_model.json')
joblib.dump(feature_cols, 'model_artifacts/feature_cols.joblib')
print('Model saved to model_artifacts/')

## 4. Deploy to Vertex AI Endpoint
Upload the model and create an endpoint for real-time prediction.

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)

# Upload model
vertex_model = aiplatform.Model.upload(
    display_name='resilientroute-risk-model',
    artifact_uri='model_artifacts/',
    serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-7:latest',
)
print(f'Model uploaded: {vertex_model.resource_name}')

# Create endpoint
endpoint = aiplatform.Endpoint.create(
    display_name='resilientroute-risk-endpoint',
)
print(f'Endpoint created: {endpoint.resource_name}')

# Deploy
vertex_model.deploy(
    endpoint=endpoint,
    machine_type='n1-standard-2',
    min_replica_count=1,
    max_replica_count=3,
)
print(f'Model deployed to endpoint!')
print(f'Endpoint ID: {endpoint.name}')

## 5. Test Prediction

In [ ]:
# Test with a sample shipment
test_instance = {
    'weather_stress': 0.85,
    'is_cyclone_season': 1,
    'port_congestion': 0.7,
    'cargo_value_cr': 12.5,
    'speed_knots': 14.0,
    'progress_pct': 42.0,
}
# Add lane one-hot features
for col in feature_cols:
    if col.startswith('lane_id_') and col not in test_instance:
        test_instance[col] = 0
test_instance['lane_id_lane-1'] = 1  # Mumbai-Singapore lane

test_df = pd.DataFrame([test_instance])[feature_cols]
pred = model.predict_proba(test_df)[:, 1][0]
print(f'Predicted delay probability: {pred:.1%}')
print(f'Risk level: {"HIGH" if pred > 0.6 else "MEDIUM" if pred > 0.3 else "LOW"}')